### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="telemonitoring_parkinsons_biomedical_voice_measurements",
    dataset_year="2007",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C59C74",
    download_description="""
We get the 2009 data from the UCI repository.

wget https://archive.ics.uci.edu/static/public/174/parkinsons.zip && unzip parkinsons.zip telemonitoring/parkinsons_updrs.data && mv telemonitoring/parkinsons_updrs.data parkinsons_updrs.data && rm -rf parkinsons.zip telemonitoring && mkdir -p local-data-warehouse/telemonitoring_parkinsons_biomedical_voice_measurements && mv parkinsons_updrs.data local-data-warehouse/telemonitoring_parkinsons_biomedical_voice_measurements/
""",
    # References
    academic_reference_bibtex="""@article{tsanas2009accurate,
  title={Accurate telemonitoring of Parkinson’s disease progression by non-invasive speech tests},
  author={Tsanas, Athanasios and Little, Max and McSharry, Patrick and Ramig, Lorraine},
  journal={Nature Precedings},
  pages={1--1},
  year={2009},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="tsanas2009accurate",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped", "WrongDomain"],
    curation_comments="""
We start with the data from UCI.

We aim to simulate the task of predicting the UPDRS score of Parkinson's patients based on their voice measurements over time. This represents the task of only getting the voice measurements to judge the UPDRS. We assume we have no measurement of a patient to make this call. Thus, we have a grouped data task, where we have to holdout entire patients over time. We thus test for one patients multiple time points and repeated measurements at once.

- We use total_UPDRS as target. Note that real UPDRS values were were obtained at baseline, three-month and six-month trial period and all other weekly cases were interpolated. Thus, the ground truth is not perfect and just an estimate based on knowing the future. We reduce it to three values (first, medium <130 days, last), as we have no other ground truth.
- We have several measurements per day for each patient when they were measured. We have some edge cases with negative test time as they got measured before the study started.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="total_UPDRS",
    problem_type="regression",
    objective_metric_name="rmse",
    group_on="subject#",
    group_time_on="test_time",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "parkinsons_updrs.data")

df = df.drop(columns=[
    "motor_UPDRS",
])
as_cat_type = ["sex", "subject#"]
df[as_cat_type] = df[as_cat_type].astype("category")

def select_rows(x):
    # Anchor times
    first_time = x.iloc[0]["test_time"]
    last_time = x.iloc[-1]["test_time"]

    before_130 = x[x["test_time"] < 130]
    last_before_130_time = before_130.iloc[-1]["test_time"]

    # Collect times to keep
    times_to_keep = {first_time, last_time}
    times_to_keep.add(last_before_130_time)

    # Keep all rows with matching test_time
    return x[x["test_time"].isin(times_to_keep)]

df = (
    df.groupby("subject#", group_keys=False)
      .apply(select_rows)
      .reset_index(drop=True)
)

df = df.sample(frac=1, random_state=42).sort_values(by=["subject#", "test_time"]).reset_index(drop=True)

/tmp/ipykernel_179732/3870508627.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("subject#", group_keys=False)
/tmp/ipykernel_179732/3870508627.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_rows)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 502
Columns: 21
Use sampling: False (sample size: 502)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['RPDE', 'NHR', 'HNR', 'DFA', 'PPE', 'Shimmer:DDA', 'Jitter(Abs)', 'Shimmer:APQ11', 'Shimmer', 'Shimmer:APQ5']
Rows remaining as candidates after top-10 filter: 0 (of 502)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,subject#,age,sex,test_time,total_UPDRS,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,Jitter:DDP,Shimmer,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
0,1,72,0,5.6431,34.398,0.00348,0.000015,0.00124,0.00133,0.00372,0.01192,0.113,0.00411,0.00463,0.00949,0.01234,0.009238,27.927,0.37340,0.52499,0.17066
1,1,72,0,5.6431,34.398,0.00662,0.000034,0.00401,0.00317,0.01204,0.02565,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
2,1,72,0,124.6500,43.524,0.00370,0.000018,0.00190,0.00185,0.00571,0.01260,0.119,0.00613,0.00617,0.01109,0.01838,0.008281,27.397,0.38173,0.53578,0.16542
3,1,72,0,124.6500,43.524,0.00264,0.000016,0.00109,0.00126,0.00327,0.01622,0.141,0.00759,0.01020,0.01431,0.02278,0.003275,27.273,0.43491,0.59390,0.13611
4,1,72,0,124.6500,43.524,0.00282,0.000015,0.00114,0.00141,0.00341,0.02343,0.235,0.01151,0.01383,0.01833,0.03452,0.005994,25.884,0.46851,0.54236,0.15182


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,subject#,category,0.0,0.0,42.0,"2, 3, 21, 37, 36, 42, 15, 22, 12, 23"
1,sex,category,0.0,0.0,2.0,"0, 1"
2,test_time,float64,0.0,0.0,122.0,"126.26, 123.83, 168.27, 122.81, 178.8, 128.76, 196.36, 124.34, 170.76, 174.66"
3,total_UPDRS,float64,0.0,0.0,132.0,"32.0, 7.0, 47.97, 36.903, 16.542, 14.734, 32.117, 16.984, 27.77, 26.826"
4,Jitter(%),float64,0.0,0.0,381.0,"0.0043, 0.0057, 0.0062, 0.0051, 0.006, 0.0041, 0.0037, 0.0036, 0.0053, 0.0043"
5,Jitter(Abs),float64,0.0,0.0,486.0,"0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0001, 0.0, 0.0"
6,Jitter:RAP,float64,0.0,0.0,303.0,"0.0018, 0.0017, 0.0018, 0.0019, 0.0019, 0.0018, 0.0014, 0.0017, 0.0018, 0.0012"
7,Jitter:PPQ5,float64,0.0,0.0,304.0,"0.0018, 0.0021, 0.0029, 0.0021, 0.0022, 0.0019, 0.0019, 0.0024, 0.0022, 0.0024"
8,Jitter:DDP,float64,0.0,0.0,419.0,"0.0054, 0.0034, 0.0036, 0.0041, 0.0095, 0.0045, 0.0057, 0.004, 0.0052, 0.005"
9,Shimmer,float64,0.0,0.0,476.0,"0.0248, 0.0142, 0.0284, 0.0138, 0.0312, 0.0566, 0.0234, 0.018, 0.0172, 0.0288"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,502.0,64.169323,8.833185,36.000000,85.000000
test_time,502.0,131.247235,57.901620,-4.262500,215.490000
total_UPDRS,502.0,29.220554,10.837527,7.000000,54.375000
Jitter(%),502.0,0.006248,0.006264,0.000830,0.080340
Jitter(Abs),502.0,0.000044,0.000038,0.000002,0.000391
Jitter:RAP,502.0,0.003034,0.003273,0.000340,0.045220
Jitter:PPQ5,502.0,0.003388,0.004499,0.000430,0.056890
Jitter:DDP,502.0,0.009102,0.009819,0.001030,0.135670
Shimmer,502.0,0.034269,0.029771,0.004550,0.268630
Shimmer(dB),502.0,0.310030,0.257014,0.041000,2.107000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column   rank                    
sex      1        0    325  64.74
         2        1    177  35.26
subject# 1        2     15   2.99
         2        3     15   2.99
         3       21     15   2.99
         4       37     14   2.79
         5       36     14   2.79

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.408,-0.809,117.452,0.17,log,4435.7,18059.7,exponential


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import GroupKFold

splits = {}
repeats = 20 # same as for IID datasets of this size.

for repeat_idx in range(repeats):
    sklearn_splits = GroupKFold(n_splits=3, random_state=42 + repeat_idx, shuffle=True).split(
        X=df,
        y=df[task_mold.target_column_name],
        groups=df[task_mold.group_on],
    )
    splits[repeat_idx] = {}
    print_once = False
    for fold_idx, (train_index, test_index) in enumerate(sklearn_splits):
        # Print len, target col count, and group counts
        train_data = df.iloc[train_index]
        test_data = df.iloc[test_index]

        if not print_once:
            print(f"""Train N: {len(train_index)}, Test N: {len(test_index)}
            Target Distribution:
            \tTrain target distribution: {df.iloc[train_index][task_mold.target_column_name].mean()}
            \tTest target distribution: {df.iloc[test_index][task_mold.target_column_name].mean()}
            Group Distribution {task_mold.group_on}:
            \tTrain: {len(train_data[task_mold.group_on].unique())}
            \tTest: {len(test_data[task_mold.group_on].unique())}
            """
            )
            print_once = True
        splits[repeat_idx][fold_idx] = (train_index.tolist(), test_index.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 3-fold split. This represents 150k (33%) customers and 1.8 million samples per test split. The label are equally represented in all train and test sets. Note, we will likely only use the first fold.",
    splits=splits,
)

Train N: 356, Test N: 146
            Target Distribution:
            	Train target distribution: 31.118519662921347
            	Test target distribution: 24.59263698630137
            Group Distribution subject#:
            	Train: 28
            	Test: 14
            
Train N: 346, Test N: 156
            Target Distribution:
            	Train target distribution: 30.05221387283237
            	Test target distribution: 27.37597435897436
            Group Distribution subject#:
            	Train: 28
            	Test: 14
            
Train N: 328, Test N: 174
            Target Distribution:
            	Train target distribution: 27.56752743902439
            	Test target distribution: 32.33660344827586
            Group Distribution subject#:
            	Train: 28
            	Test: 14
            
Train N: 338, Test N: 164
            Target Distribution:
            	Train target distribution: 28.447437869822487
            	Test target distribution: 30.81392682926829
     

## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7b94-bbb0-756e-a312-0cb02efb7c7e
1918ffc04c1ab76b4b3178d19c67cf7751f111c7f5a022ca29a291e51649eed3
